# Step caching (first-block cache) on compile + Sage fp16-PV: Wan2.1-T2V-1.3B on L4

Runtime: L4. Predictions to check: `predictions_stepcache_2026-11-13.md` (C1–C9). All runs share this session.

Time: about 10 min to build SageAttention, then about 75 min of runs (~3 compute units).

**Before running:** the fp32 reference frames are uploaded instead of regenerated (saves 28 min). The upload cell asks for two files from your computer: `frames_0.npy` and `frames_3.npy` in `video-dit-l4/results/fp32_33f_s3/` (80 MB). They are also in the earlier `sage_results (1).zip`. The cell checks their hashes.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total,clocks.max.sm,power.limit --format=csv

In [ ]:
!pip install -q diffusers==0.42.1 transformers==5.22.2 accelerate ftfy imageio imageio-ffmpeg lpips 2>&1 | tail -5
import torch, diffusers, transformers
print(torch.__version__, 'cuda', torch.version.cuda, diffusers.__version__, transformers.__version__)

In [ ]:
%cd /content
!git -C video-dit-l4 pull || git clone https://github.com/FredWen-S/video-dit-l4.git
%cd /content/video-dit-l4

## fp32 reference frames (upload, ~2 min)

In [ ]:
import glob, os, re
d = 'results/fp32_33f_s3'
os.makedirs(d, exist_ok=True)
if len(glob.glob(d + '/frames_*.npy')) < 2:
    from google.colab import files
    for name, data in files.upload().items():   # browsers may rename to "frames_0 (1).npy"
        m = re.match(r'frames_(\d+)', name)
        if m:
            open(f'{d}/frames_{m.group(1)}.npy', 'wb').write(data)
import subprocess, sys
r = subprocess.run([sys.executable, 'bench/hash_frames.py', d, '--check', 'results/fp32_33f/frames_sha256.json'],
                   capture_output=True, text=True)
print(r.stdout, r.stderr)
if r.returncode:
    raise RuntimeError('fp32 reference frames missing or not the expected ones; upload frames_0.npy and frames_3.npy')

## Build SageAttention for sm_89 (~10 min)

Same build as `05_sage_attention.ipynb`, pinned to the commit measured there. Stops the notebook if it fails.

In [ ]:
import os, re, subprocess, sys, torch
maj, mi = torch.version.cuda.split('.')[:2]
m = re.search(r'release (\d+)\.(\d+)', subprocess.run('nvcc --version', shell=True, capture_output=True, text=True).stdout)
print('torch CUDA', torch.version.cuda, '| nvcc', m.group(0) if m else 'not found')
if not m or m.group(1) != maj:
    ubuntu = subprocess.run('. /etc/os-release && echo ubuntu${VERSION_ID/./}', shell=True, capture_output=True,
                            text=True, executable='/bin/bash').stdout.strip()
    v = f'{maj}-{mi}'
    !wget -q https://developer.download.nvidia.com/compute/cuda/repos/{ubuntu}/x86_64/cuda-keyring_1.1-1_all.deb && dpkg -i cuda-keyring_1.1-1_all.deb > /dev/null
    !apt-get -qq update > /dev/null && apt-get -qq install -y cuda-minimal-build-{v} cuda-cudart-dev-{v} cuda-cccl-{v} cuda-nvrtc-dev-{v} libcublas-dev-{v} libcusparse-dev-{v} libcusolver-dev-{v} libcurand-dev-{v} > /dev/null
    os.environ['CUDA_HOME'] = f'/usr/local/cuda-{maj}.{mi}'
    os.environ['PATH'] = os.environ['CUDA_HOME'] + '/bin:' + os.environ['PATH']
    !nvcc --version | tail -2

%cd /content
!git clone -q https://github.com/thu-ml/SageAttention.git || true
!git -C SageAttention checkout -q d1a57a5  # the commit measured in results/L4_sage.md
!git -C SageAttention log -1 --format='SageAttention %h %cd'
%cd /content/SageAttention
# torch >= 2.10 headers use C++20 `requires` clauses; SageAttention's setup.py hardcodes -std=c++17
!sed -i 's/-std=c++17/-std=c++20/g' setup.py && grep -n 'std=c++' setup.py && g++ --version | head -1
!TORCH_CUDA_ARCH_LIST=8.9 EXT_PARALLEL=4 MAX_JOBS=8 pip install --no-build-isolation -v . > /content/sage_build.log 2>&1; tail -40 /content/sage_build.log
%cd /content/video-dit-l4
r = subprocess.run([sys.executable, '-c', 'import importlib.metadata as m, sageattention; print("sageattention", m.version("sageattention"))'],
                   capture_output=True, text=True)
print(r.stdout, r.stderr[-2000:])
if r.returncode != 0:
    raise RuntimeError('SageAttention did not build or import; stopping here. Send the output above.')

## 33 frames, 4 prompts

Uncached base (~12 min). C1 checks it against `sage_fp16pv_33f`.

In [ ]:
!python bench/profile_baseline.py --frames 33 --compile blocks --attn-backend _sage_qk_int8_pv_fp16_cuda --out results/pv16_33f
!python bench/hash_frames.py results/pv16_33f --check results/sage_fp16pv_33f/frames_sha256.json
!python bench/compare_frames.py --ref results/fp32_33f_s3 --test results/pv16_33f --out-name compare_vs_fp32.json

First-block cache at three thresholds (~6–9 min each). Each run is compared with the uncached base and with fp32.

In [ ]:
for thr, name in [(0.05, 'fbc005'), (0.10, 'fbc010'), (0.20, 'fbc020')]:
    out = f'results/{name}_33f'
    !python bench/profile_baseline.py --frames 33 --compile blocks --attn-backend _sage_qk_int8_pv_fp16_cuda --cache-threshold {thr} --out {out}
    !python bench/compare_frames.py --ref results/pv16_33f --test {out}
    !python bench/compare_frames.py --ref results/fp32_33f_s3 --test {out} --out-name compare_vs_fp32.json

Control: fewer steps, no cache (~8 and ~6 min). 30 steps = 60% of the forwards, 20 steps = 40%.

In [ ]:
for steps in [30, 20]:
    out = f'results/steps{steps}_33f'
    !python bench/profile_baseline.py --frames 33 --compile blocks --attn-backend _sage_qk_int8_pv_fp16_cuda --steps {steps} --out {out}
    !python bench/compare_frames.py --ref results/pv16_33f --test {out}
    !python bench/compare_frames.py --ref results/fp32_33f_s3 --test {out} --out-name compare_vs_fp32.json

CLIP score from the exact frames, same source for every run (C9, ~5 min).

In [ ]:
!python bench/clip_score.py results/pv16_33f results/fbc005_33f results/fbc010_33f results/fbc020_33f results/steps30_33f results/steps20_33f --source npy

## 81 frames, fox (~15 min), C8

In [ ]:
B81 = '--frames 81 --n-prompts 1 --compile blocks --attn-backend _sage_qk_int8_pv_fp16_cuda'
!python bench/profile_baseline.py {B81} --out results/pv16_81f
!python bench/profile_baseline.py {B81} --cache-threshold 0.10 --out results/fbc010_81f
!python bench/compare_frames.py --ref results/pv16_81f --test results/fbc010_81f

In [ ]:
# Hashes of every run; zip without .npy
import glob, os
runs = sorted(d for d in glob.glob('results/pv16_*') + glob.glob('results/fbc*') + glob.glob('results/steps*') if os.path.isdir(d))
for d in runs:
    !python bench/hash_frames.py {d}
runs_str = ' '.join(runs)
!zip -qr cache_results.zip {runs_str} -x '*.npy' && ls -lh cache_results.zip
from google.colab import files; files.download('cache_results.zip')